# 01 — Inside a Seurat object

**CIAD single-cell workshop**

Before starting the analysis, it is worth knowing what you are holding.

A Seurat object is a container. Counts (number of transcripts/molecules per gene across cells) and every result you compute —
QC metrics, clusters, UMAP coordinates, cell type labels — are stored inside
the same object. That is convenient,
but it means the object gradually fills with things whose location is not obvious.

This notebook opens it up. Same data as the rest of the workshop: 2,700 PBMCs
from 10x Genomics.

**What you should be able to do afterwards**

- say what an assay, a layer, and a reduction are
- pull out the counts, the metadata, the cell names, the gene names
- add your own column of metadata
- subset an object without breaking it
- find where a result was stored, without guessing

About 30 minutes. Run cells with *Shift + Enter*.

## Setup

The first line installs the packages. The `library()` lines after it are
the normal way to load a package in R.

In [ ]:
# This line installs the packages we need. It is a shortcut for the
# workshop, so that nobody spends the class waiting for an install.
source("https://raw.githubusercontent.com/MartinLoza/CIAD_workshop_sc/main/setup/setup.R")

# This is the normal way to load a package in R. You will write lines
# like these at the top of every script you make.
library(Seurat)
library(ggplot2)
library(dplyr)
library(patchwork)

# Size of every figure in this notebook, in inches. Change these two
# numbers if a plot looks too small or too large.
options(repr.plot.width = 10, repr.plot.height = 7)

## 1. Before the object: the matrix

Everything starts as a matrix of counts. Genes in rows, cells in columns, and
each entry is the number of molecules of that gene detected in that cell.

We download that matrix from 10x Genomics. It comes as three files, which is the
standard 10x output.

- `matrix.mtx` — the counts themselves
- `barcodes.tsv` — one line per cell
- `genes.tsv` — one line per gene

In [ ]:
url <- "https://cf.10xgenomics.com/samples/cell/pbmc3k/pbmc3k_filtered_gene_bc_matrices.tar.gz"

download.file(url, "pbmc3k.tar.gz", quiet = TRUE)
untar("pbmc3k.tar.gz")

list.files("filtered_gene_bc_matrices/hg19")

### What is inside those files

Before loading anything, look at the files themselves. They are plain text.

In [ ]:
dir <- "filtered_gene_bc_matrices/hg19"

# readLines(n = 3) stops after 3 lines; matrix.mtx has millions of them
cat("barcodes.tsv — one line per cell\n")
writeLines(readLines(file.path(dir, "barcodes.tsv"), n = 3))

cat("\ngenes.tsv — one line per gene\n")
writeLines(readLines(file.path(dir, "genes.tsv"), n = 3))

cat("\nmatrix.mtx — the counts\n")
writeLines(readLines(file.path(dir, "matrix.mtx"), n = 6))

Three things to notice.

- **`barcodes.tsv`** holds cell barcodes, such as `AAACATACAACCAC-1`. A barcode
  is a short DNA sequence that was attached to everything coming from one
  droplet. It is simply the name of the cell.
- **`genes.tsv`** has two columns: the Ensembl gene id, and the gene symbol we
  read (`MS4A1`, `CD3E`). Seurat uses the symbol.
- **`matrix.mtx`** starts with two header lines. The next line gives three
  numbers: how many genes, how many cells, and how many counts are not zero.
  Every line after that is one count, written as *gene number, cell number,
  count*.

Multiply the first two numbers and compare with the third. The full table would
hold about 88 million values, and the file stores only a few million of them.

`Read10X()` reads those three files into one matrix: genes in rows, cells in
columns.

In [ ]:
counts <- Read10X(data.dir = "filtered_gene_bc_matrices/hg19")

dim(counts)

32,738 genes and 2,700 cells. Let us look at a corner of it — five genes,
five cells.

In [ ]:
counts[1:5, 1:5]

All zeros, which is not a mistake. Pick genes that are actually expressed
and the picture changes.

In [ ]:
counts[c("CD3D", "CD3E", "MS4A1", "CD14", "LYZ"), 1:5]

### Why the dots

Notice the `.` instead of `0`.

This is not an ordinary matrix. It is a **sparse** matrix — class `dgCMatrix` —
which stores only the non-zero entries. A dot means "nothing stored here", which
is to say zero.

That matters more than it sounds. Most entries in single-cell data are zero,
and storing them all would be wasteful to the point of being impossible.

In [ ]:
cat("class:", class(counts), "\n\n")

zeros <- 1 - length(counts@x) / (nrow(counts) * ncol(counts))
cat(sprintf("entries that are zero: %.1f%%\n\n", 100 * zeros))

cat("sparse :", format(object.size(counts), units = "MB"), "\n")
cat("dense  :", format(object.size(as.matrix(counts[1:2000, ])) * (nrow(counts) / 2000),
                      units = "MB"), "(estimated)\n")

## 2. Making the object

`CreateSeuratObject()` wraps that matrix into something that can carry results.

In [ ]:
pbmc <- CreateSeuratObject(
  counts       = counts,
  project      = "pbmc3k",
  min.cells    = 3,
  min.features = 200
)

pbmc

Read that summary line by line. It says: one **assay**, `RNA`, which is the
**active** one; 13,714 features; 2,700 samples; one **layer**, called `counts`.

Fewer genes than we started with — `min.cells = 3` dropped genes seen in almost
no cells.

### The vocabulary

| term | what it means |
|---|---|
| **assay** | one measurement type over the same cells — here `RNA`. A CITE-seq experiment would also have `ADT` for surface proteins. Integration adds a corrected assay alongside the original. |
| **layer** | one version of the matrix inside an assay: `counts` is raw, `data` normalised, `scale.data` centred and scaled. Same cells, same genes, different numbers. (Seurat v4 called these *slots*.) |
| **metadata** | one row per cell, one column per thing you know about it: QC metrics, sample of origin, cluster, cell type. |
| **reduction** | a low-dimensional representation — PCA, UMAP. One row per cell, a handful of columns. |
| **identity** | the cell's current label. Whatever `Idents()` returns. This is what plots colour by and what differential expression compares. |

The rest of this notebook is where each of those lives and how to get at it.

## 3. Assays and layers

An assay is a named box. List them, and see which is active.

In [ ]:
cat("assays        :", Assays(pbmc), "\n")
cat("active assay  :", DefaultAssay(pbmc), "\n")
cat("layers in RNA :", Layers(pbmc[["RNA"]]), "\n")

One layer so far, because we have not normalised. Do that and a second
appears — the raw counts are not replaced.

In [ ]:
pbmc <- NormalizeData(pbmc, verbose = FALSE)

Layers(pbmc[["RNA"]])

### Getting the matrix out

`LayerData()` is the accessor. Name the assay and the layer you want.

Reaching in with `@` also works and you will see it in older code, but the
internals changed between Seurat v4 and v5 — the accessor did not. Use the
accessor.

In [ ]:
raw  <- LayerData(pbmc, assay = "RNA", layer = "counts")
norm <- LayerData(pbmc, assay = "RNA", layer = "data")

genes <- c("CD3D", "MS4A1", "CD14", "LYZ", "PPBP")
cells <- colnames(pbmc)[1:5]

cat("--- counts (raw molecules) ---\n")
print(raw[genes, cells])

cat("\n--- data (log-normalised) ---\n")
print(round(norm[genes, cells], 2))

Same genes, same cells, two layers. The raw layer holds whole numbers —
molecules counted. The normalised layer holds decimals, corrected for how much
RNA each cell gave up and then log-transformed.

Zeros stay zero throughout. Normalisation does not invent expression.

## 4. Metadata

One row per cell. This is where everything you know *about* cells lives, as
opposed to what they express.

In [ ]:
dim(pbmc@meta.data)

head(pbmc@meta.data, 5)

Three columns already, and we added none of them:

- `orig.ident` — the `project` name we passed in
- `nCount_RNA` — total molecules in the cell
- `nFeature_RNA` — genes detected in the cell

The last two were computed by `CreateSeuratObject()` because they are needed so
often.

### Two ways to read a column

`pbmc$name` is the short form. `pbmc[["name"]]` returns a data frame instead of a
vector — occasionally what you want, usually not.

In [ ]:
summary(pbmc$nFeature_RNA)

cat("\nvector      :", class(pbmc$nCount_RNA), "\n")
cat("data frame  :", class(pbmc[["nCount_RNA"]]), "\n")

### Adding a column

Assign to `$` with one value per cell, in the order the cells are in.

In [ ]:
pbmc$percent.mt <- PercentageFeatureSet(pbmc, pattern = "^MT-")

# anything of the right length works — here, a crude size label
pbmc$size <- ifelse(pbmc$nCount_RNA > median(pbmc$nCount_RNA), "large", "small")

head(pbmc@meta.data, 3)

table(pbmc$size)

### ✏️ Exercise 1

Add a column called `high.mito` that is `TRUE` when a cell's mitochondrial
percentage is above 5, and `FALSE` otherwise. Then count how many cells that is.

Fill in the blank:

In [ ]:
# pbmc$high.mito <- ______

table(pbmc$high.mito)

## 5. Cells and genes

Names, not numbers, are how you address things.

In [ ]:
cat("cells:", ncol(pbmc), " genes:", nrow(pbmc), "\n\n")

cat("first 3 cell names:\n"); print(head(Cells(pbmc), 3))
cat("\nfirst 3 gene names:\n"); print(head(Features(pbmc), 3))

Cell names are the 10x barcodes — the DNA tag that identified the droplet.
They are the object's row names in the metadata and its column names in the
matrix, and Seurat keeps those aligned for you.

`colnames()` and `rownames()` work too and mean the same thing.

In [ ]:
identical(Cells(pbmc), colnames(pbmc))
identical(Features(pbmc), rownames(pbmc))

## 6. Identities

The identity is the cell's current label — what plots colour by, and what
differential expression uses as its grouping.

Right now every cell has the same one, because nothing has been clustered.

In [ ]:
head(Idents(pbmc), 3)

table(Idents(pbmc))

You can set identities from any metadata column, and put them back. This is
worth practising, because it is the single most common way people confuse
themselves later.

In [ ]:
Idents(pbmc) <- "size"          # use the column we invented
table(Idents(pbmc))

# keep a copy before overwriting — identities are easy to lose
pbmc$my_grouping <- Idents(pbmc)

Idents(pbmc) <- "orig.ident"    # and back
table(Idents(pbmc))

## 7. Where results go

To show reductions and graphs we need to have computed some. This cell runs the
standard pipeline quickly and silently — notebook 02 is where we explain what
each step does and why. For now, watch what it adds to the object.

In [ ]:
pbmc <- FindVariableFeatures(pbmc, verbose = FALSE)
pbmc <- ScaleData(pbmc, verbose = FALSE)
pbmc <- RunPCA(pbmc, verbose = FALSE)
pbmc <- FindNeighbors(pbmc, dims = 1:10, verbose = FALSE)
pbmc <- FindClusters(pbmc, resolution = 0.5, verbose = FALSE)
pbmc <- RunUMAP(pbmc, dims = 1:10, verbose = FALSE)

pbmc

Compare that summary to the one from section 2. The object now reports three
layers, and two dimensional reductions.

Everything landed inside the object. Nothing was returned to a loose variable.

In [ ]:
cat("layers     :", Layers(pbmc[["RNA"]]), "\n")
cat("reductions :", Reductions(pbmc), "\n")
cat("graphs     :", Graphs(pbmc), "\n")
cat("metadata   :", paste(colnames(pbmc@meta.data), collapse = ", "), "\n")

Note `seurat_clusters` appeared in the metadata on its own — `FindClusters()`
writes it there, and also sets it as the identity.

### Reductions

A reduction is a matrix: one row per cell, one column per dimension.
`Embeddings()` gets the cell coordinates.

In [ ]:
pca  <- Embeddings(pbmc, reduction = "pca")
umap <- Embeddings(pbmc, reduction = "umap")

cat("pca  :", nrow(pca),  "cells x", ncol(pca),  "components\n")
cat("umap :", nrow(umap), "cells x", ncol(umap), "dimensions\n\n")

round(umap[1:5, ], 3)

Those two numbers per cell are the whole UMAP plot. `DimPlot()` draws them,
but there is nothing magic underneath — you could plot them yourself.

In [ ]:
p1 <- DimPlot(pbmc, reduction = "umap") + ggtitle("DimPlot")

df <- as.data.frame(umap)
colnames(df) <- c("dim1", "dim2")   # whatever Seurat named them
df$cluster <- pbmc$seurat_clusters

p2 <- ggplot(df, aes(dim1, dim2, colour = cluster)) +
  geom_point(size = 0.3) +
  ggtitle("the same numbers, plotted by hand")

p1 + p2

`Loadings()` gets the other half of a PCA: how much each **gene** contributes
to each component.

In [ ]:
loadings <- Loadings(pbmc, reduction = "pca")

cat(nrow(loadings), "genes x", ncol(loadings), "components\n\n")
round(loadings[1:5, 1:3], 3)

### ✏️ Exercise 2

Which five genes pull hardest on PC 1, in either direction?

Fill in the blank — you want the largest values by absolute size:

In [ ]:
pc1 <- Loadings(pbmc, reduction = "pca")[, 1]

# head(sort(______, decreasing = TRUE), 5)

## 8. Subsetting

`subset()` takes cells, genes, or both, and keeps everything consistent —
metadata, layers and reductions are all cut down together.

In [ ]:
b_cells <- subset(pbmc, subset = seurat_clusters == 3)

cat("original :", ncol(pbmc),    "cells\n")
cat("subset   :", ncol(b_cells), "cells\n\n")

# the metadata came along
cat("metadata rows:", nrow(b_cells@meta.data), "\n")
# so did the UMAP
cat("umap rows    :", nrow(Embeddings(b_cells, "umap")), "\n")

You can subset on any metadata column, on expression of a gene, or on
identity.

In [ ]:
# by expression
cd3_pos <- subset(pbmc, subset = CD3E > 1)
cat("CD3E-positive cells:", ncol(cd3_pos), "\n")

# by identity — several clusters at once
few <- subset(pbmc, idents = c(0, 1, 2))
cat("clusters 0,1,2     :", ncol(few), "cells\n")

# by gene, keeping all cells
small <- subset(pbmc, features = VariableFeatures(pbmc)[1:100])
cat("100 genes          :", nrow(small), "genes,", ncol(small), "cells\n")

### ✏️ Exercise 3

Make an object containing only cells that are **not** in cluster 3, and confirm
the numbers add up to the original.

Fill in the blank:

In [ ]:
# not_b <- subset(pbmc, subset = ______)

# ncol(not_b) + ncol(b_cells) == ncol(pbmc)

## 9. A map of the object

Where to look for things, in one place.

| what you want | how to get it |
|---|---|
| the raw counts | `LayerData(obj, assay = "RNA", layer = "counts")` |
| the normalised values | `LayerData(obj, assay = "RNA", layer = "data")` |
| all metadata | `obj@meta.data` |
| one metadata column | `obj$column` |
| cell names | `Cells(obj)` or `colnames(obj)` |
| gene names | `Features(obj)` or `rownames(obj)` |
| current labels | `Idents(obj)` |
| which assays exist | `Assays(obj)` |
| which layers exist | `Layers(obj[["RNA"]])` |
| which reductions exist | `Reductions(obj)` |
| UMAP or PCA coordinates | `Embeddings(obj, reduction = "umap")` |
| gene contributions to PCs | `Loadings(obj, reduction = "pca")` |
| variable genes | `VariableFeatures(obj)` |
| how many cells / genes | `ncol(obj)` / `nrow(obj)` |

Two habits worth keeping:

- **Use the accessors, not `@`.** The internals changed between v4 and v5 and may
  change again. `LayerData()` did not.
- **Put anything you care about in the metadata.** `Idents()` is one vector and
  is overwritten constantly. A metadata column survives.

Next: notebook 02, where we actually do the analysis — quality control,
clustering, and naming the cell types.

---

### Answers

<details>
<summary>Click to expand</summary>

**Exercise 1**

```r
pbmc$high.mito <- pbmc$percent.mt > 5
```

**Exercise 2**

```r
head(sort(abs(pc1), decreasing = TRUE), 5)
```

`abs()` is the point: a large negative loading matters as much as a large
positive one. The sign only says which end of the component the gene pushes
towards.

**Exercise 3**

```r
not_b <- subset(pbmc, subset = seurat_clusters != 3)
ncol(not_b) + ncol(b_cells) == ncol(pbmc)
```

</details>